In [ ]:
# ============================================
# CalRatio Trigger Efficiency Analysis
# Benchmark: (mH, mS) = (1000, 475) GeV
#            cτ = 6.04 m
# Author: ALHADHUR Ahlam
# Supervisors: Dr. Louie CORPE, 
#              Dr. Marion MISSIO
# LPCA, Université Clermont Auvergne
# June 2026
# ============================================

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import uproot
import awkward as ak 
import os 
import matplotlib.ticker as ticker 

In [ ]:
Root_file= "\\Users\\ahlam\\Internship\\Fichier.root\\HSS\\outputFiles6.root"

# Binning configuration for histograms and efficiciency plots 
pt_bins= np.linspace(0, 500, 26)
Lxy_bins = np.linspace(0, 5.0, 26)
Lz_bins = np.linspace(0, 7.0, 26)   

# ATLAS style configuration for plots 

ATLAS_STYLE ={
    "font.family":"sans-serif",
    "font.size":12,
    "axes.labelsize":14,
    "axes.titlesize":13,
    "axes.linewidth":1.2,
    "axes.facecolor":"white",
    "xtick.labelsize":12,
    "ytick.labelsize":12,
    "xtick.direction":"in",
    "ytick.direction":"in",
    "xtick.top":True,
    "ytick.right":True,
    "xtick.minor.visible":True,
    "ytick.minor.visible":True,
    "xtick.major.size":8,
    "xtick.minor.size":4,
    "ytick.major.size":8,
    "ytick.minor.size":4,
    "xtick.major.width":1.2,
    "ytick.major.width":1.2,
    "figure.facecolor":"white",
    "legend.frameon": True,
    "legend.framealpha": 1.0,
    "legend.edgecolor": "black",
    "legend.fontsize":10,
    "lines.linewidth":1.5,
    "lines.markersize":6
}
plt.rcParams.update(ATLAS_STYLE)



In [ ]:
# Configured the  family of triggers to analyze based on the provided Root file and the specified trigger family name (e.g., "calratio", "calratiormbib", "calratiovar"). 
# It identifies the relevant triggers in the ROOT file and categorizes them accordingly.
with uproot.open(Root_file) as f:
    triggers = [k for k in f["reco"].keys() if "CLEANllp" in k]
    triggers_calratio=[k for k in triggers if "calratio_" in k.lower()
                       and "rmbib" not in k.lower()
                          and "var" not in k.lower()]
    triggers_calratiormbib=[k for k in triggers if "calratiormbib" in k.lower()]
    triggers_calratiovar=[k for k in triggers if "calratiovar" in k.lower()]

## Functions

In [ ]:
# Function to shorten the trigger names for better visualization in the plots 
# by removing common prefixes and truncating long names
def short_trigger_name(trig):
    short = trig.replace("trigPassed_HLT_j30_CLEANllp_momemfrac006_", "")
    short = short.replace("trigPassed_HLT_j30_CLEANllp_momemfrac012_", "")
    short = short.replace("trigPassed_", "")
    return short if len(short) < 30 else short[:27] + "..."

# Function to extract the L1 seed from the trigger name by looking 
# for the substring "_L1" and returning the part of the name after it, prefixed with "L1"
def get_L1_seed(trig_name):              
    if "_L1" in trig_name:
        return "L1" + trig_name.split("_L1")[-1]
    return trig_name

# add ATLAS labels to the plots with the option to include signal and ctau information,
#  with configurable position and styling
def add_atlas_labels(ax, signal_label=None, ctau_label=None,   
                     x=0.05, y=0.95):
    ax.text(x, y,
            r"$\bf{ATLAS}$ Simulation",    
            transform=ax.transAxes,
            fontsize=13,
            verticalalignment="top")

    if signal_label is not None:
        ax.text(x, y - 0.07,
                signal_label,
                transform=ax.transAxes,
                fontsize=11,
                verticalalignment="top")

    if ctau_label is not None:
        ax.text(x, y - 0.14,
                ctau_label,
                transform=ax.transAxes,
                fontsize=11,
                verticalalignment="top")

In [ ]:
# Loading the trees from the Root file
def load_tree(filepath):
    f=uproot.open(filepath)
    return f["particleLevel"], f["reco"]

# Loanding variables from the trees and appling geometic cuts for the calorimeter acceptance
def load_variables(tree_truth, tree_reco):
    evt_truth= ak.to_numpy(tree_truth["eventNumber"].array(library="ak"))
    evt_reco = ak.to_numpy(tree_reco["eventNumber"].array(library="ak"))
    pt_raw = tree_truth["truth_alp_pt"].array(library="ak")
    eta_raw = tree_truth["truth_alp_eta"].array(library="ak")
    phi_raw = tree_truth["truth_alp_phi"].array(library="ak")
    Lx_raw = tree_truth["truth_alp_decayVtxX"].array(library="ak")
    Ly_raw = tree_truth["truth_alp_decayVtxY"].array(library="ak")
    Lz_raw = tree_truth["truth_alp_decayVtxZ"].array(library="ak")


    Lxy_all = np.sqrt(Lx_raw**2 + Ly_raw**2)/1000.0  
    Lz_all= np.abs(Lz_raw)/1000.0
    eta_all=np.abs(eta_raw)

    # Geometric cuts for each ALP 
    in_barrel_all= (eta_all < 1.4) & (Lxy_all > 1.5) & (Lxy_all < 3.9)
    in_endcap_all= (eta_all > 1.4) & (Lz_all > 3.5) & (Lz_all < 6.0)
    in_calo_alp= in_barrel_all| in_endcap_all

    # Create a mask for events where at least one ALP is in the calorimeter acceptance 
    pt_mask= ak.where(in_calo_alp, pt_raw, 0.0)
    idx_max= ak.argmax(pt_mask,axis=1, keepdims=True)

    
    pt= ak.to_numpy(ak.fill_none(ak.max(pt_mask, axis=1), 0.0))/1000.0
    eta= ak.to_numpy(ak.fill_none(ak.firsts(eta_raw[idx_max], axis=1), 0.0))
    phi= ak.to_numpy(ak.fill_none(ak.firsts(phi_raw[idx_max], axis=1), 0.0))
    Lx= ak.to_numpy(ak.fill_none(ak.firsts(Lx_raw[idx_max], axis=1), 0.0))
    Ly= ak.to_numpy(ak.fill_none(ak.firsts(Ly_raw[idx_max], axis=1), 0.0))
    Lz= ak.to_numpy(ak.fill_none(ak.firsts(Lz_raw[idx_max], axis=1), 0.0))

    Lxy= np.sqrt(Lx**2 + Ly**2)/1000.0
    Lz= np.abs(Lz)/1000.0

    df_truth= pd.DataFrame({
        "eventNumber": evt_truth,
        "pt": pt, "eta": eta, "phi": phi, "Lxy": Lxy, "Lz": Lz
    })

    
    df_reco = pd.DataFrame({"eventNumber": evt_reco})
    for trig in triggers:
        df_reco[trig] = ak.to_numpy(tree_reco[trig].array(library="ak")).astype(bool)

    df= pd.merge(df_truth,df_reco,on="eventNumber", how="inner")

    trig_col= [c for c in df.columns if "CLEANllp" in c]
    df["pass_any_trig"]= df[trig_col].any(axis=1)

    # Geometric cuts for the colorimater 
    df["in_barrel"]= (np.abs(df["eta"]) < 1.4) & (df["Lxy"] > 1.5) & (df["Lxy"] < 3.9)
    df["in_endcap"]= (np.abs(df["eta"]) > 1.4) & (df["Lz"] > 3.5) & (df["Lz"] < 6.0)
    df["in_calo"]= df["in_barrel"] | df["in_endcap"]

    

    return df

In [ ]:
# Function to compute the DeltaR between the truth ALP and the reco jets and 
# to check if there is a match within a givin DeltaR cut 
def compute_DeltaR(df, tree_reco, dR_cut=0.2):

    jet_eta= tree_reco["EMTopoJet_eta"].array(library="ak")
    jet_phi= tree_reco["EMTopoJet_phi"].array(library="ak")
    evt_reco= ak.to_numpy(tree_reco["eventNumber"].array(library="ak"))

    evt_to_idx= {evt: idx for idx, evt in enumerate(evt_reco)}
    llp_eta= df["eta"].to_numpy()
    llp_phi= df["phi"].to_numpy()

    matched=[]

    for i in range(len(df)):
        evt= df["eventNumber"].iloc[i]
        reco_idx= evt_to_idx.get(evt,-1)

        if reco_idx == -1:
            matched.append(False)
            continue

        jets_eta_i = ak.to_numpy(jet_eta[reco_idx])
        jets_phi_i = ak.to_numpy(jet_phi[reco_idx])
        
        if len(jets_eta_i) == 0:
            matched.append(False)
            continue

        deta= jets_eta_i - llp_eta[i]
        dphi= jets_phi_i - llp_phi[i]

        dphi= (dphi + np.pi) % (2 * np.pi) - np.pi
        dR= np.sqrt(deta**2 + dphi**2)
        matched .append(bool(np.min(dR) < dR_cut))


    df= df.copy()
    df["matched_jet"]=matched 
    return df   

In [ ]:
# Compute the efficiency for a given variable and a given mask (if any) by appling 
# the appropriate geometric cuts and matching criteria for the calorimeter acceptance and the reco jets 
def compute_efficiency(df, variable,bins, mask=None):
    
    calo_mask= df["in_calo"]
    if mask is not None:
        calo_mask= calo_mask & mask 

    denom_mask = calo_mask & df["matched_jet"]
    num_mask= denom_mask & df["pass_any_trig"]
    
    h_before,_=np.histogram(df.loc[denom_mask, variable], bins=bins)
    h_after,_=np.histogram(df.loc[num_mask, variable],bins=bins)

    eff= np.divide(h_after,h_before, out=np.zeros_like(h_after, dtype=float), where=h_before>0)
    err = np.sqrt(np.divide(eff * (1 - eff), h_before,out=np.zeros_like(eff),where=h_before > 0))

    return eff, err, h_before,h_after

In [ ]:
# Compute the efficiency for a family of triggers (e.g. all triggers with calRatio) by appling
#  the appropriate geometric cuts and matching criteria for the calorimeter acceptance and the reco jets 
def eff_family(df, trig_list, bins, variable, mask=None):  

    calo_mask = df["in_calo"]
    if mask is not None:
        calo_mask = calo_mask & mask

    denom_mask = calo_mask & df["matched_jet"]
    family_OR = df[trig_list].any(axis=1)
    num_mask = denom_mask & family_OR

    h_before, _ = np.histogram(
        df.loc[denom_mask, variable], bins=bins)   
    h_after,  _ = np.histogram(
        df.loc[num_mask,   variable], bins=bins)  

    eff = np.divide(h_after, h_before,
                    out=np.zeros_like(h_before, dtype=float),
                    where=h_before > 0)
    err = np.sqrt(np.divide(eff * (1 - eff), h_before,
                            out=np.zeros_like(eff),
                            where=h_before > 0))

    eff = np.where(h_before <= 5, np.nan, eff)
    err = np.where(h_before <= 5, np.nan, err)

    eff_global = h_after.sum() / h_before.sum() \
                 if h_before.sum() > 0 else 0

    return eff, err, eff_global

### Plot function 

In [ ]:
# Function to plot the efficiency as a function of a given variable with error bars and save the figure
def plot_efficiency(eff,err,bins, xlabel, output_file, label="Trigger OR"):

    centers = 0.5 * (bins[:-1] + bins[1:])

    plt.figure(figsize=(8, 6))
    plt.errorbar(centers, eff, yerr=err,
                 marker="o",
                 linestyle="-",
                 color="blue",
                 markersize=5,
                 label=label)

    plt.xlabel(xlabel)
    plt.ylabel("Efficiency")
    plt.ylim(0, 1.4)
    plt.xlim(bins[0], bins[-1])
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    plt.show()
    print(f"Figure sauvegardée: {output_file}")

In [ ]:
# Plot the efficiency comparison between different families of triggers (e.g. calRatio, calRatioRmBiB, calRatioVar) 
# and the OR of all triggers as a functions of a given variable with error bars and save the figure  
def plot_comparison_families(df, bins, variable, xlabel, output_file,
                              mask=None, 
                              major_tick=None,                   
                              signal_label=r"$(m_H, m_S)=(600, 150)$ GeV",
                              ctau_label=r"$c\tau_{gen}=1.84$ m"):

    centers = 0.5 * (bins[:-1] + bins[1:])   
    trig_cols = [c for c in df.columns if "CLEANllp" in c]

    # --- Compute efficiencies for each family ---
    eff_cr, err_cr,  eff_g_cr = eff_family(df, triggers_calratio,
                                              bins, variable, mask)
    eff_rb, err_rb,  eff_g_rb = eff_family(df, triggers_calratiormbib,
                                              bins, variable, mask)
    eff_var, err_var, eff_g_var = eff_family(df, triggers_calratiovar,
                                              bins, variable, mask)

    # --- OR via compute_efficiency ---
    eff_or, err_or, h_before, h_after = compute_efficiency(
        df, variable, bins, mask=mask)          
    eff_g_or = h_after.sum()/ h_before.sum() \
               if h_before.sum() > 0 else 0

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    ax.errorbar(centers, eff_cr,  yerr=err_cr,
                marker="o", linestyle="-", color="red",
                capsize=2, label="calRatio")

    ax.errorbar(centers, eff_rb,  yerr=err_rb,
                marker="s", linestyle="-", color="steelblue",
                capsize=2, label="calRatioRmBiB")

    ax.errorbar(centers, eff_var, yerr=err_var,
                marker="D", linestyle="-", color="green",
                capsize=2, label="calRatioVar")

    ax.errorbar(centers, eff_or,  yerr=err_or,
                marker="^", linestyle="--", color="black",
                capsize=2, linewidth=2,
                label="OR all triggers")

    add_atlas_labels(ax, signal_label=signal_label,
                     ctau_label=ctau_label)

    ax.set_xlabel(xlabel, fontsize=13)         
    ax.set_ylabel("Efficiency", fontsize=13)
    ax.set_title("", fontsize=13)
    ax.set_ylim(0, 1.4)
    ax.set_xlim(bins[0], bins[-1])
    ax.legend(fontsize=10, loc="upper right")
    ax.minorticks_on()
    ax.tick_params(direction="in", top=True, right=True)

    if major_tick is not None:
        ax.xaxis.set_major_locator(
            ticker.MultipleLocator(major_tick))
        ax.xaxis.set_minor_locator(
            ticker.MultipleLocator(major_tick / 5))

    plt.tight_layout()
    plt.savefig(output_file, dpi=300,
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Figure saved: {output_file}")

In [ ]:
# plot function to plot the efficiency of the OR of all triggers as a function of a given variable with error bars and save the figure.
def plot_trigger_OR(eff, err, bins, xlabel, output_file,
                    signal_label= r"$(m_H, m_S)=(600, 150)$ GeV",
                    ctau_label=r"$c\tau_{gen}=1.84$ m",
                    major_tick=None):
        
        centers= 0.5*(bins[:-1] + bins[1:])
        eff_plot=np.nan_to_num(eff,nan=0.0)
        err_plot=np.nan_to_num(err,nan=0.0)


        fig, ax= plt.subplots(figsize=(8,6))

        ax.fill_between(centers,0,eff_plot,alpha=0.15,color="gold",linewidth=0,step="mid")
        ax.fill_between(centers,
                        eff_plot-err_plot,
                        eff_plot+err_plot,
                        alpha=0.25,
                        color="gold",
                        linewidth=0,step="mid"
                        )
        ax.errorbar(centers, eff, yerr=err,
                    marker="s",linestyle="-",
                    linewidth=1.5, color="darkorange",markersize=6,capsize=2,
                    label="Trigger OR")
        
        add_atlas_labels(ax, signal_label=signal_label, ctau_label=ctau_label)

        ax.set_xlabel(xlabel,fontsize=14)
        ax.set_ylabel("Efficiency", fontsize=14)
        ax.set_ylim(0, 1.4)
        ax.set_xlim(bins[0], bins[-1])

        if major_tick is not None:
                ax.xaxis.set_major_locator(ticker.MultipleLocator(major_tick))
                ax.xaxis.set_minor_locator(ticker.MultipleLocator(major_tick/5))

        ax.minorticks_on()
        ax.tick_params(direction="in", top=True, right=True)
        ax.legend(loc="upper right")


        plt.tight_layout()
        plt.savefig(output_file,dpi= 300)
        plt.show()
        print(f"Figure saved: {output_file}")


# Function to plot the efficiency of each trigger in a given family (e.g. all triggers with calRatio) 
# as a function of a given variable with error bars and save the figure
def plot_eff_by_L1_seed(df, bins, variable, xlabel, output_file,  
                         trigger_list, family_name,
                         mask=None,                                 
                         signal_label=r"$(m_H, m_S)=(600, 150)$ GeV",
                         ctau_label=r"$c\tau_{gen}=1.84$ m"):

    centers = 0.5 * (bins[:-1] + bins[1:])
    trigger_list = [t for t in trigger_list if t in df.columns]
    if len(trigger_list) == 0:
        print("No triggers found.")
        return

    calo_mask = df["in_calo"]
    if mask is not None:
        calo_mask = calo_mask & mask                               

    denom_mask = calo_mask & df["matched_jet"]
    h_before, _ = np.histogram(df.loc[denom_mask, variable],bins=bins)
    if h_before.sum() == 0:
        print("No events in denominator.")
        return

    colors = ["red", "black", "crimson",
                  "steelblue", "seagreen", "purple", "blue"]
    linestyles = ["-", "--", "-.", ":", "-", "--", "-."]
    markers = ["o", "s", "D", "^", "v", "P", "X"]
    filled = [True, True, False, True, False, True, False]

    fig, ax = plt.subplots(figsize=(10, 6))

    for i, trig in enumerate(trigger_list):
        mask_num = denom_mask & df[trig]
        h_after, _ = np.histogram(df.loc[mask_num,variable],     
                                   bins=bins)

        eff = np.divide(h_after, h_before,
                        out=np.zeros_like(h_before, dtype=float),
                        where=h_before > 0)
        err = np.sqrt(np.divide(eff * (1 - eff), h_before,
                                out=np.zeros_like(eff),
                                where=h_before > 0))

        if np.nanmax(eff) < 1e-6:
            continue

        color = colors[i % len(colors)]
        mfc = color if filled[i % len(filled)] else "none"

        ax.errorbar(centers, eff, yerr=err,
                    marker=markers[i % len(markers)],
                    linestyle=linestyles[i % len(linestyles)],
                    linewidth=1.5, color=color,
                    markersize=6, markerfacecolor=mfc,
                    markeredgecolor=color, capsize=2,
                    label="L1 seed: " + get_L1_seed(trig))

    add_atlas_labels(ax, signal_label, ctau_label)

    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel("Efficiency", fontsize=14)
    ax.set_ylim(0, 1.4)
    ax.set_xlim(bins[0], bins[-1])
    ax.minorticks_on()
    ax.tick_params(direction="in", top=True, right=True)
    ax.legend(fontsize=8, loc="upper right", ncol=1,
              bbox_to_anchor=(1.0, 1.0),
              frameon=True, edgecolor="black")
    ax.set_title(family_name, fontsize=12)

    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Figure sauvegardée : {output_file}")

In [ ]:
# Function to plot the efficiency of each trigger in a given family ( e.g. all triggers with calRatio) as function 
# of a given variable with error bars and save the figure, with subplots for the different geometric regions ( barrel vs endcap)
def plot_eff_by_L1_seed_subplots(df, output_file,
                                   trigger_list, family_name,
                                   major_tick=None,
                                   signal_label=r"$(m_H, m_S)=(600, 150)$ GeV",
                                   ctau_label=r"$c\tau_{gen}=1.84$ m"):

    trigger_list = [t for t in trigger_list if t in df.columns]
    if len(trigger_list) == 0:
        print(f"No triggers found for {family_name}")
        return

    colors = ["red", "black","crimson",
                  "steelblue","seagreen","purple", "blue"]
    linestyles = ["-", "--", "-.", ":", "-", "--", "-."]
    markers = ["o", "s", "D", "^", "v", "P", "X"]
    filled = [True, True, False, True, False, True, False]

    # Dénominateurs
    denom_pt = df["in_calo"] & df["matched_jet"]
    denom_barrel = df["in_calo"] & df["in_barrel"] & df["matched_jet"]
    denom_endcap = df["in_calo"] & df["in_endcap"] & df["matched_jet"]

    h_before_pt, _ = np.histogram(df.loc[denom_pt, "pt"], bins=pt_bins)
    h_before_Lxy,_ = np.histogram(df.loc[denom_barrel,"Lxy"], bins=Lxy_bins)
    h_before_Lz, _ = np.histogram(df.loc[denom_endcap,"Lz"],bins=Lz_bins)

    centers_pt = 0.5 * (pt_bins[:-1] + pt_bins[1:])
    centers_Lxy = 0.5 * (Lxy_bins[:-1]+ Lxy_bins[1:])
    centers_Lz = 0.5 * (Lz_bins[:-1]+ Lz_bins[1:])

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor("white")

    titles = [f"{family_name}- vs $p_T$",
               f"{family_name}- vs $L_{{xy}}$ (barrel)",
               f"{family_name} - vs $L_z$ (endcap)"]
    xlabels = ["LLP $p_T$ [GeV]",
               "LLP $L_{xy}$ [m]",
               "LLP $L_{z}$ [m]"
]
    for i, trig in enumerate(trigger_list):

        color = colors[i % len(colors)]
        mfc = color if filled[i % len(filled)] else "none"
        label = "L1 seed: " + get_L1_seed(trig)

        # --- vs pT ---
        h_after_pt, _ = np.histogram(
            df.loc[denom_pt & df[trig], "pt"], bins=pt_bins)
        eff_pt = np.divide(h_after_pt, h_before_pt,
                           out=np.zeros_like(h_before_pt, dtype=float),
                           where=h_before_pt > 0)
        err_pt = np.sqrt(np.divide(eff_pt * (1 - eff_pt), h_before_pt,
                                   out=np.zeros_like(eff_pt),
                                   where=h_before_pt > 0))

        # --- vs Lxy ---
        h_after_Lxy, _ = np.histogram(
            df.loc[denom_barrel & df[trig], "Lxy"], bins=Lxy_bins)
        eff_Lxy = np.divide(h_after_Lxy, h_before_Lxy,
                            out=np.zeros_like(h_before_Lxy, dtype=float),
                            where=h_before_Lxy > 0)
        err_Lxy = np.sqrt(np.divide(eff_Lxy * (1 - eff_Lxy), h_before_Lxy,
                                    out=np.zeros_like(eff_Lxy),
                                    where=h_before_Lxy > 0))

        # --- vs Lz ---
        h_after_Lz, _ = np.histogram(
            df.loc[denom_endcap & df[trig], "Lz"], bins=Lz_bins)
        eff_Lz = np.divide(h_after_Lz, h_before_Lz,
                           out=np.zeros_like(h_before_Lz, dtype=float),
                           where=h_before_Lz > 0)
        err_Lz = np.sqrt(np.divide(eff_Lz * (1 - eff_Lz), h_before_Lz,
                                   out=np.zeros_like(eff_Lz),
                                   where=h_before_Lz > 0))

        for ax, centers, eff, err in zip(
            axes,
            [centers_pt, centers_Lxy, centers_Lz],
            [eff_pt, eff_Lxy, eff_Lz],
            [err_pt, err_Lxy, err_Lz]
        ):
            ax.errorbar(centers, eff, yerr=err,
                        marker=markers[i % len(markers)],
                        linestyle=linestyles[i % len(linestyles)],
                        linewidth=1.5, color=color,
                        markersize=6, markerfacecolor=mfc,
                        markeredgecolor=color, capsize=2,
                        label=label)

   
    for ax, title, xlabel, bins in zip(
        axes,
        titles,
        xlabels,
        [pt_bins, Lxy_bins, Lz_bins]
    ):
        ax.set_facecolor("white")
        add_atlas_labels(ax, signal_label=signal_label,
                         ctau_label=ctau_label)
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel("Efficiency", fontsize=12)
        ax.set_ylim(0, 1.4)
        ax.set_xlim(bins[0], bins[-1])
        ax.minorticks_on()
        ax.tick_params(direction="in", top=True, right=True)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=7, loc="upper right",
                  frameon=True, edgecolor="black")
        if major_tick is not None:
            ax.xaxis.set_major_locator(
                ticker.MultipleLocator(major_tick))
            ax.xaxis.set_minor_locator(
                ticker.MultipleLocator(major_tick / 5))

    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300,
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Figure saved: {output_file}")

In [ ]:
# Function to plot the distribution of a given variable for the numerator and denominator of a given family of triggers 
# (e.g. all triggers with calRatio) with subplots for the different geometric regions ( barrel vs endcap) and save the figure
def plot_hist_num_denom(df, output_file,
                                  trigger_list,
                                  family_name,
                                  major_tick=None,
                                  signal_label=r"$(m_H, m_S)=(600, 150)$ GeV",
                                  ctau_label=r"$c\tau_{gen}=1.84$ m"):

    trigger_list = [t for t in trigger_list if t in df.columns]

    calo_mask = df["in_calo"] & df["in_barrel"]
    denom_mask = calo_mask & df["matched_jet"]
    family_OR = df[trigger_list].any(axis=1)
    num_mask = denom_mask & family_OR

    centers_Lxy = 0.5 * (Lxy_bins[:-1] + Lxy_bins[1:])

    h_denom, _ = np.histogram(df.loc[denom_mask, "Lxy"], bins=Lxy_bins)
    h_num, _ = np.histogram(df.loc[num_mask,"Lxy"], bins=Lxy_bins)
    
    fig, axes = plt.subplots(1,3, figsize=(21, 6))
    fig.patch.set_facecolor("white")
    for ax in axes:
        ax.set_facecolor("white")

    
    axes[0].step(centers_Lxy, h_denom,
                 color="steelblue", linewidth=2, alpha=0.8,
                 label="in calo & matched jet")
    axes[0].fill_between(centers_Lxy, 0, h_denom,
                         step="pre", color="steelblue", alpha=0.15)
    add_atlas_labels(axes[0], signal_label=signal_label,
                     ctau_label=ctau_label)
    axes[0].set_xlabel("LLP $L_{xy}$ [m]", fontsize=14)
    axes[0].set_ylabel("Events", fontsize=14)
    axes[0].set_xlim(Lxy_bins[0], Lxy_bins[-1])
    axes[0].set_ylim(0)
    axes[0].set_title(f"Denominator vs $L_{{xy}}$ (barrel) - {family_name}", fontsize=12)
    axes[0].minorticks_on()
    axes[0].tick_params(direction="in", top=True, right=True)
    axes[0].legend(fontsize=10, loc="upper right")

    axes[1].step(centers_Lxy, h_num,
                 color="darkorange", linewidth=2, alpha=0.8,
                 label="in calo & matched jet ")
    axes[1].fill_between(centers_Lxy, 0, h_num,
                         step="pre", color="darkorange", alpha=0.15)
    add_atlas_labels(axes[1], signal_label=signal_label,
                     ctau_label=ctau_label)
    axes[1].set_xlabel("LLP $L_{xy}$ [m]", fontsize=14)
    axes[1].set_ylabel("Events", fontsize=14)
    axes[1].set_xlim(Lxy_bins[0], Lxy_bins[-1])
    axes[1].set_ylim(0)
    axes[1].set_title(f"Numerator vs $L_{{xy}}$ (barrel) - {family_name}",
                      fontsize=12)
    axes[1].minorticks_on()
    axes[1].tick_params(direction="in", top=True, right=True)
    axes[1].legend(fontsize=10, loc="upper right")

    
    axes[2].step(centers_Lxy, h_denom,
                 color="steelblue", linewidth=2, alpha=0.8,
                 label="Denominator : in calo & matched jet")
    axes[2].fill_between(centers_Lxy, 0, h_denom,
                         step="pre", color="steelblue", alpha=0.15)
    axes[2].step(centers_Lxy, h_num,
                 color="darkorange", linewidth=2, alpha=0.8,
                 label="Numerator")
    axes[2].fill_between(centers_Lxy, 0, h_num,
                         step="pre", color="darkorange", alpha=0.15)
    add_atlas_labels(axes[2], signal_label=signal_label,
                     ctau_label=ctau_label)
    axes[2].set_xlabel("LLP $L_{xy}$ [m]", fontsize=14)
    axes[2].set_ylabel("Events", fontsize=14)
    axes[2].set_xlim(Lxy_bins[0], Lxy_bins[-1])
    axes[2].set_ylim(0)
    axes[2].set_title(f" Superposition of $L_{{xy}}$ distribution - {family_name}", fontsize=12)
    axes[2].minorticks_on()
    axes[2].tick_params(direction="in", top=True, right=True)
    axes[2].legend(fontsize=10, loc="upper right")
    
    if major_tick is not None:
        ax.xaxis.set_major_locator(ticker.MultipleLocator(major_tick))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(major_tick/5))

        ax.minorticks_on()
        ax.tick_params(direction="in", top=True, right=True)
        ax.legend(loc="upper right")


    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Figure saved: {output_file}")